# Домашнє завдання: Внесення оновлень в БД і робота з транзакціями

Це ДЗ передбачене під виконання на локальній машині. Виконання з Google Colab буде суттєво ускладнене.

## Підготовка
1. Переконайтесь, що у вас встановлены необхідні бібліотеки:
   ```bash
   pip install sqlalchemy pymysql pandas matplotlib seaborn python-dotenv
   ```

2. Створіть файл `.env` з параметрами підключення до бази даних classicmodels. Базу даних ви можете отримати через

  - docker-контейнер згідно існтрукції в [документі](https://www.notion.so/hannapylieva/Docker-1eb94835849480c9b2e7f5dc22ee4df9), також відео інструкції присутні на платформі - уроки "MySQL бази, клієнт для роботи з БД, Docker і ChatGPT для запитів" та "Як встановити Docker для роботи з базами даних без терміналу"
  - або встановивши локально цю БД - для цього перегляньте урок "Опціонально. Встановлення MySQL та  БД Сlassicmodels локально".
  
  Приклад `.env` файлу ми створювали в лекції. Ось його обовʼязкове наповнення:
    ```
    DB_HOST=your_host
    DB_PORT=3306 або 3307 - той, який Ви налаштували
    DB_USER=your_username
    DB_PASSWORD=your_password
    DB_NAME=classicmodels
    ```
  Якщо ви створили цей файл під час перегляду лекції - **новий створювати не треба**. Замініть лише назву БД, або пропишіть назву в коді створення підключення (замість отримання назви цільової БД зі змінних оточення). Але переконайтесь, що до `.env` файл лежить в тій самій папці, що і цей ноутбук.

  **УВАГА!** НЕ копіюйте скрит для **створення** `.env` файлу. В лекції він наводиться для прикладу. І давалось пояснення, що в реальних проєктах ми НІКОЛИ не пишемо доступи до бази в коді. Копіювання скрипта для створення `.env` файлу сюди в ДЗ буде вважатись грубою помилкою і ми зніматимемо бали.

3. Налаштуйте підключення через SQLAlchemy до БД за прикладом в лекції.

Рекомендую вивести (відобразити) змінну engine після створення. Вона має бути не None! Якщо None - значить у Вас не підтягнулись налаштування з .env файла.

Ви також можете налаштувати параметри підключення до БД без .env файла, просто прописавши текстом в відповідних місцях. Це - не рекомендований підхід.


## Завдання

### Завдання 1: Оновлення інформації про клієнта (2 бали)

**Створіть функцію для оновлення контактної інформації клієнта за його номером** з наступними можливостями:
- Оновлення телефону клієнта
- Оновлення email (якщо поле існує в таблиці)

Опціонально, якщо вам хочеться більше практики:
- Логування змін в окрему таблицю

Використайте підхід з параметризованими запитами через `text()` та `UPDATE` оператор. Не забудьте на початку перевірити чи існує клієнт з таким номером в базі - це хороша практика.

Отримати всі колонки, які існують в таблиці ви можете наступним запитом
```
  SELECT COLUMN_NAME, DATA_TYPE
  FROM INFORMATION_SCHEMA.COLUMNS
  WHERE TABLE_NAME = 'customers'
```

Запустіть функцію і продемонструйте її роботу, запустивши SELECT, який допоможе це зробити.



In [19]:
from sqlalchemy import text
import pandas as pd

def update_customer_contact(engine, customer_number, new_phone=None, new_email=None):
    
    with engine.connect() as conn:
        
        # Перевіряємо чи існує клієнт
        check_query = text("""
            SELECT customerNumber, customerName
            FROM customers
            WHERE customerNumber = :cust_no
        """)
        
        result = conn.execute(check_query, {"cust_no": customer_number}).fetchone()
        
        if result is None:
            print("Клієнта з таким номером не існує.")
            return
        
        print(f"Знайдено клієнта: {result.customerName}")
        
        
        # Оновлюємо телефон та email клієнта
        update_contacts = []
        params = {"cust_no": customer_number}
        
        if new_phone:
            update_contacts.append("phone = :phone")
            params["phone"] = new_phone
            
        if new_email:
            update_contacts.append("email = :email")
            params["email"] = new_email
        
        if not update_contacts:
            print("Немає даних для оновлення")
            return
        
        update_query = text(f"""
            UPDATE customers
            SET {", ".join(update_contacts)}
            WHERE customerNumber = :cust_no
        """)
        
        conn.execute(update_query, params)
        
        print("Дані клієнта оновлено!")

In [16]:
update_customer_contact(
    engine,
    customer_number=100,
    new_phone="+65 224 1222"
)

Клієнта з таким номером не існує.


In [20]:
update_customer_contact(
    engine,
    customer_number=166,
    new_phone="+65 224 1222"
)

Знайдено клієнта: Handji Gifts& Co
Дані клієнта оновлено!


In [21]:
query = """
SELECT customerNumber, customerName, phone
FROM customers
WHERE customerNumber = 166
"""

df_check = pd.read_sql(query, engine)
display(df_check)

,customerNumber,customerName,phone
0,166,Handji Gifts& Co,+65 224 1222


### Завдання 2: Створення нового замовлення з транзакцією (5 балів)

**Реалізуйте процес створення нового замовлення** з наступними кроками в одній транзакції:
- Створення запису в таблиці `orders`
- Додавання товарних позицій в `orderdetails`
- Перевірка наявності товарів на складі
- Зменшення кількості товарів на складі

Запустіть процес з тестовими даними і продемонструйте через SELECT, що процес успішно відпрацював і були виконані необхідні операції.




In [23]:
def create_order(engine, customer_number, items):

    with engine.begin() as conn:

        # Створення запису в таблиці orders
        result = conn.execute(text("SELECT MAX(orderNumber) FROM orders"))
        order_number = result.scalar() + 1
        conn.execute(text("""
            INSERT INTO orders 
            (orderNumber, orderDate, requiredDate, shippedDate, status, comments, customerNumber)
            VALUES (:orderNumber, :orderDate, :requiredDate, :shippedDate, 'In Process', 'Check on availability', :customerNumber)
        """), {
            "orderNumber": order_number,
            "orderDate": date.today(),
            "requiredDate": date.today(),
            "shippedDate" : date.today(),
            "customerNumber": customer_number
        })
        line = 1
       
        for item in items:

            product_code = item["productCode"]
            quantity = item["quantity"]

        # Перевірка наявності товарів на складі
            stock_query = text("""
                SELECT quantityInStock, buyPrice
                FROM products
                WHERE productCode = :code
            """)

            product = conn.execute(stock_query, {"code": product_code}).fetchone()

            if product.quantityInStock < quantity:
                print("Недостатньо товару на складі")
                return

            price = product.buyPrice

         # Додавання товарних позицій в orderdetails
            conn.execute(text("""
                INSERT INTO orderdetails
                (orderNumber, productCode, quantityOrdered, priceEach, orderLineNumber)
                VALUES (:orderNumber, :code, :qty, :price, :line)
            """), {
                "orderNumber": order_number,
                "code": product_code,
                "qty": quantity,
                "price": price,
                "line": line
            })

        #З меншення кількості товарів на складі
            conn.execute(text("""
                UPDATE products
                SET quantityInStock = quantityInStock - :qty
                WHERE productCode = :code
            """), {
                "qty": quantity,
                "code": product_code
            })

            line += 1

        print("Замовлення створено:", order_number)

In [25]:
from datetime import date
from sqlalchemy import text
items = [
    {"productCode": "S10_1678", "quantity": 3},
    {"productCode": "S10_1949", "quantity": 5}
]
create_order(engine, 103, items)

Замовлення створено: 10426


In [27]:
pd.read_sql("""
SELECT *
FROM orders
ORDER BY orderNumber DESC
LIMIT 5
""", engine)

,orderNumber,orderDate,requiredDate,shippedDate,status,comments,customerNumber
0,10426,2026-03-06,2026-03-06,2026-03-06,In Process,Check on availability,103
1,10425,2005-05-31,2005-06-07,None,In Process,None,119
2,10424,2005-05-31,2005-06-08,None,In Process,None,141
3,10423,2005-05-30,2005-06-05,None,In Process,None,314
4,10422,2005-05-30,2005-06-11,None,In Process,None,157
